In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import sys
import numpy.random as rand

rand.seed(1993)

sys.path.append('/home/austin/Basic')
from utils_np import cosineSimilarity

In [ ]:
coefs_list = []
comp_list = []
nSamps = 19
aucs1 = np.zeros(nSamps)
aucs2 = np.zeros(nSamps)
aucs3 = np.zeros(nSamps)
recons = np.zeros(nSamps)

S_list = []

x = np.arange(2,21)

for i in range(nSamps):
    pname = 'Unbalanced_Elastic_12_enc_1.0_' + str(int(i+2)) + '.p'
    myDict = pickle.load(open(pname,'rb'))
    aucs1[i] = myDict['rTest1_0']
    aucs2[i] = myDict['rTest2_0']
    aucs3[i] = myDict['rTest3_0']
    coefs_list.append(myDict['A_enc'])
    comp_list.append(myDict['components'])
    S_list.append(myDict['S_train'])

# We first look at the raw predictive ability of the models, the only "identifiable" metric of comparison

In [ ]:
fs1 = 16
fs2 = 24
plt.plot(x,aucs1)
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Holdout Mouse AUC',fontsize=fs1)
plt.title('Prediction Mouse 048',fontsize=fs2)

In [ ]:
plt.plot(x,aucs2)
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Holdout Mouse AUC',fontsize=fs1)
plt.title('Prediction Mouse 7980',fontsize=fs2)

In [ ]:
plt.plot(aucs3)
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Holdout Mouse AUC',fontsize=fs1)
plt.title('Prediction Mouse 7998',fontsize=fs2)

### Compare predictive performance across latent dimensions

Evaluate prediction and reconstruction separately. Stability of prediction alone does not establish that every fitted parameter is unchanged.

# Now we look at the predictive coefficients themselves. How do the predictive networks change based on the number of generative components?

In [ ]:
similarities = [np.zeros(nSamps) for _ in range(nSamps)]
for i in range(nSamps):
    for j in range(nSamps):
        similarities[i][j] = cosineSimilarity(coefs_list[i][:,0],coefs_list[j][:,0])

In [ ]:
plt.plot(x,similarities[0])
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Encoder Similarity',fontsize=fs1)
plt.title('Model with 2 Factors',fontsize=fs2)

In [ ]:
plt.plot(x,similarities[6])
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Encoder Similarity',fontsize=fs1)
plt.title('Model with 8 Factors',fontsize=fs2)

In [ ]:
nRep = 10
base_line_similarities = np.zeros((nSamps,nRep))

for i in range(nSamps):
    for j in range(nRep):
        choices = np.random.choice(i+2,size=2,replace=False)
        ii = choices[0]
        jj = choices[1]
        A_enc = coefs_list[i]
        a1 = A_enc[:,ii]
        a2 = A_enc[:,jj]
        base_line_similarities[i,j] = cosineSimilarity(a1,a2)

baseline_mean = np.mean(base_line_similarities,axis=1)
baseline_std = np.std(base_line_similarities,axis=1)

In [ ]:
plt.plot(x,baseline_mean)
plt.xlabel('Number of Components',fontsize=fs1-1)
plt.ylabel('Random Network Similarity',fontsize=fs1-1)
plt.title('How Similar are Encoder Networks?',fontsize=fs2-4)

### Compare supervised encoders with randomly selected networks

Cosine similarity measures alignment; it is not a significance test. The manuscript reports a three-network exception to otherwise similar supervised encoders.

# Now we look at the components. How much does the predictive network change based on the number of chosen components?

In [ ]:
similarities_components = [np.zeros(nSamps) for _ in range(nSamps)]
for i in range(nSamps):
    for j in range(nSamps):
        similarities_components[i][j] = cosineSimilarity(comp_list[i][0],comp_list[j][0])

In [ ]:
plt.plot(x,similarities_components[0])
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Decoder Similarity',fontsize=fs1)
plt.title('Model with 2 Factors',fontsize=fs2)

In [ ]:
plt.plot(x,similarities_components[6])
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Decoder Similarity',fontsize=fs1)
plt.title('Model with 8 Factors',fontsize=fs2)

In [ ]:
plt.plot(x,similarities_components[13])
plt.xlabel('Number of Components',fontsize=fs1)
plt.ylabel('Decoder Similarity',fontsize=fs1)
plt.title('Model with 15 Factors',fontsize=fs2)

In [ ]:
nRep = 10
base_line_component_similarities = np.zeros((nSamps,nRep))

for i in range(nSamps):
    for j in range(nRep):
        choices = np.random.choice(i+2,size=2,replace=False)
        ii = choices[0]
        jj = choices[1]
        comp = comp_list[i]
        a1 = comp[ii]
        a2 = comp[jj]
        base_line_component_similarities[i,j] = cosineSimilarity(a1,a2)

baseline_mean_comp = np.mean(base_line_component_similarities,axis=1)
baseline_std_comp = np.std(base_line_component_similarities,axis=1)

In [ ]:
plt.plot(x,baseline_mean_comp)
plt.xlabel('Number of Components',fontsize=fs1-1)
plt.ylabel('Random Network Similarity',fontsize=fs1-1)
plt.title('How Similar are Decoder Networks?',fontsize=fs2-4)

#### Basically similarity drops substantially

In [ ]:
my_dict = {'base_line_component_similarities':base_line_component_similarities,
          'similarities_components':similarities_components,
          'similarities':similarities,
          'base_line_similarities':base_line_similarities}
import pickle
pickle.dump(my_dict,open('Results.p','wb'))